# SD3.5 Batch Image Generation

In [ ]:
# Install packages
!pip -q install -U diffusers transformers accelerate sentencepiece safetensors huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 116.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from google.colab import userdata
from huggingface_hub import login
import getpass

token = userdata.get("HF_TOKEN")

if not token:
    token = getpass.getpass("Enter your Hugging Face token: ")

login(token=token)

In [ ]:
# Set paths and generation settings
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
PROJECT_ROOT = "YOUR_PROJECT_ROOT"
# CSV_PATH = f"{PROJECT_ROOT}/finetune_prompts/van_gogh_speculative_finetune_prompts-150.csv"
CSV_PATH = f"{PROJECT_ROOT}/finetune_prompts/van_gogh_speculative_finetune_prompts-300.csv"
# OUTPUT_DIR = f"{PROJECT_ROOT}/sd35_outputs-150"
OUTPUT_DIR = f"{PROJECT_ROOT}/sd35_outputs-300"

WIDTH = 1024
HEIGHT = 1024
GUIDANCE_SCALE = 7.0
NUM_INFERENCE_STEPS = 28
BASE_SEED = 42

In [ ]:
# Load the CSV
import pandas as pd

required_cols = ["concept_rank", "concept_name", "prompt_id", "prompt"]

df = pd.read_csv(CSV_PATH)
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df[required_cols].copy()
df.head()


,concept_rank,concept_name,prompt_id,prompt
0,1,Star-swept night skies and cypress silhouettes,1,A tall cypress rising beside a moonlit village...
1,1,Star-swept night skies and cypress silhouettes,2,Three dark cypress trees leaning over a hillsi...
2,1,Star-swept night skies and cypress silhouettes,3,A small chapel tower beneath curling constella...
3,1,Star-swept night skies and cypress silhouettes,4,Rooftops clustered in a basin of blue hills wi...
4,1,Star-swept night skies and cypress silhouettes,5,A winding lane descending toward lit windows u...


In [ ]:
# Load the model
import torch
from diffusers import StableDiffusion3Pipeline

print(torch.cuda.is_available())

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

pipe = StableDiffusion3Pipeline.from_pretrained(MODEL_ID, torch_dtype=dtype)
pipe.set_progress_bar_config(disable=True)


if torch.cuda.is_available():
    pipe.enable_model_cpu_offload()
else:
    pipe.to("cpu")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


True


model_index.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [ ]:
# Generate and save images
import re
from pathlib import Path

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

def slugify(value):
    value = re.sub(r"[^a-zA-Z0-9_-]+", "_", str(value).strip())
    value = value.strip("_")
    return value[:80] or "item"

records = []

device = "cuda" if torch.cuda.is_available() else "cpu"

for i, row in df.iterrows():
    concept_rank = int(row["concept_rank"])
    concept_name = str(row["concept_name"])
    prompt_id = str(row["prompt_id"])
    prompt = str(row["prompt"])

    filename = f"{concept_rank:02d}_{slugify(concept_name)}_{slugify(prompt_id)}.png"
    image_path = out_dir / filename

    generator = torch.Generator(device=device).manual_seed(BASE_SEED + i)

    image = pipe(
        prompt=prompt,
        width=WIDTH,
        height=HEIGHT,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        generator=generator,
    ).images[0]

    image.save(image_path)
    records.append(
        {
            "concept_rank": concept_rank,
            "concept_name": concept_name,
            "prompt_id": prompt_id,
            "prompt": prompt,
            "image_path": str(image_path),
        }
    )
    print(f"[{i + 1}/{len(df)}] {image_path.name}")


[1/300] 01_Star-swept_night_skies_and_cypress_silhouettes_1.png
[2/300] 01_Star-swept_night_skies_and_cypress_silhouettes_2.png
[3/300] 01_Star-swept_night_skies_and_cypress_silhouettes_3.png
[4/300] 01_Star-swept_night_skies_and_cypress_silhouettes_4.png
[5/300] 01_Star-swept_night_skies_and_cypress_silhouettes_5.png
[6/300] 01_Star-swept_night_skies_and_cypress_silhouettes_6.png
[7/300] 01_Star-swept_night_skies_and_cypress_silhouettes_7.png
[8/300] 01_Star-swept_night_skies_and_cypress_silhouettes_8.png
[9/300] 01_Star-swept_night_skies_and_cypress_silhouettes_9.png
[10/300] 01_Star-swept_night_skies_and_cypress_silhouettes_10.png
[11/300] 01_Star-swept_night_skies_and_cypress_silhouettes_11.png
[12/300] 01_Star-swept_night_skies_and_cypress_silhouettes_12.png
[13/300] 01_Star-swept_night_skies_and_cypress_silhouettes_13.png
[14/300] 01_Star-swept_night_skies_and_cypress_silhouettes_14.png
[15/300] 01_Star-swept_night_skies_and_cypress_silhouettes_15.png
[16/300] 01_Star-swept_night

In [ ]:
# Save a manifest
manifest_path = out_dir / "manifest.csv"
pd.DataFrame(records).to_csv(manifest_path, index=False)

In [ ]:
# Convert manifest.csv -> metadata.jsonl
import json

metadata_path = out_dir / "metadata.jsonl"

with open(metadata_path, "w", encoding="utf-8") as f:
    for _, row in pd.read_csv(manifest_path).iterrows():
        f.write(json.dumps({
            "file_name": Path(row["image_path"]).name,
            "text": row["prompt"],
        }, ensure_ascii=False) + "\n")